In [22]:
#====================================
#Celda 1 — Imports y configuración
#====================================
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../../data/economico")
print(f"📁 Ruta de datos: {DATA_PATH.resolve()}")


📁 Ruta de datos: /workspaces/TFG_Spain-s_Migratory_Flow/data/economico


In [ ]:
#=============================================
#Celda 2 — Cargar todos los JSONs disponibles
#=============================================

# Cargar todos los JSONs acumulados
archivos = sorted(glob.glob(str(DATA_PATH / "**/*.json"), recursive=True))
print(f"📦 JSONs encontrados: {len(archivos)}")
for f in archivos:
    print(f"  → {f}")

# Cargar el más reciente
with open(archivos[-1], "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n✅ JSON cargado: {archivos[-1]}")
print(f"🕐 Timestamp: {data.get('timestamp')}")
print(f"🔑 Claves disponibles: {list(data.keys())}")



📦 JSONs encontrados: 5
  → ../../data/economico/2026/03/19_03_2026.json
  → ../../data/economico/2026/03/20_03_2026.json
  → ../../data/economico/2026/03/23_03_2026.json
  → ../../data/economico/2026/03/24_03_2026.json
  → ../../data/economico/2026/03/25_03_2026.json

✅ JSON cargado: ../../data/economico/2026/03/25_03_2026.json
🕐 Timestamp: 2026-03-25T10:24:44.990932
🔑 Claves disponibles: ['timestamp', 'tasa_paro_ccaa', 'renta_media_hogar', 'densidad_poblacion', 'turismo_viajeros', 'turismo_pernoctaciones', 'centros_educativos_ccaa', 'ipv_precio_vivienda_ccaa', 'ipv_precio_vivienda_anual', 'compraventas_vivienda', 'hipotecas_vivienda', 'crimen', 'centros_salud', 'catastro', 'alquiler_ministerio']


In [24]:
#=============================================
#Celda 3 — Explorar tasa de paro por CCAA
#=============================================
# La API INE devuelve lista de objetos con Nombre, MetaData y Data
paro_raw = data["tasa_paro_ccaa"]
print(f"Tipo: {type(paro_raw)} | Registros: {len(paro_raw)}")
print(f"\nEstructura de un registro:")
print(json.dumps(paro_raw[0], indent=2, ensure_ascii=False))


Tipo: <class 'list'> | Registros: 420

Estructura de un registro:
{
  "COD": "EPA86913",
  "Nombre": "Tasa de paro de la población. Ambos sexos. Total Nacional. Todas las edades. ",
  "T3_Unidad": "Tasas",
  "T3_Escala": " ",
  "MetaData": [
    {
      "Id": 283910,
      "T3_Variable": "Tipo de dato",
      "Nombre": "Tasa de paro de la población",
      "Codigo": ""
    },
    {
      "Id": 454,
      "T3_Variable": "Sexo",
      "Nombre": "Ambos sexos",
      "Codigo": ""
    },
    {
      "Id": 16473,
      "T3_Variable": "Total Nacional",
      "Nombre": "Total Nacional",
      "Codigo": "00"
    },
    {
      "Id": 15668,
      "T3_Variable": "Totales de edad",
      "Nombre": "Total",
      "Codigo": ""
    }
  ],
  "Data": [
    {
      "Fecha": "2023-10-01T00:00:00.000+02:00",
      "T3_TipoDato": "Definitivo",
      "T3_Periodo": "T4",
      "Anyo": 2023,
      "Valor": 11.76
    },
    {
      "Fecha": "2023-07-01T00:00:00.000+02:00",
      "T3_TipoDato": "Definitivo",
  

In [25]:
#=============================================
#Celda 4 — Parsear tasa de paro a DataFrame
#=============================================
def parse_ine_tabla(raw: list) -> pd.DataFrame:
    """Convierte la respuesta JSON de INE API a DataFrame limpio"""
    filas = []
    for serie in raw:
        nombre = serie.get("Nombre", "")
        metadata = serie.get("MetaData", [])
        datos = serie.get("Data", [])
        
        # Extraer dimensiones del MetaData
        dims = {m.get("Variable", {}).get("Nombre", f"dim_{i}"): m.get("Nombre", "")
                for i, m in enumerate(metadata)}
        
        for punto in datos:
            fila = {**dims}
            fila["valor"] = punto.get("Valor")
            fila["periodo"] = punto.get("NombrePeriodo")
            fila["fecha"] = punto.get("Fecha")
            fila["secreto"] = punto.get("Secreto", False)
            filas.append(fila)
    
    return pd.DataFrame(filas)

df_paro = parse_ine_tabla(paro_raw)
print(f"Shape: {df_paro.shape}")
print(f"\nColumnas: {list(df_paro.columns)}")
print(f"\nPrimeras filas:")
df_paro.head(10)


Shape: (1680, 8)

Columnas: ['dim_0', 'dim_1', 'dim_2', 'dim_3', 'valor', 'periodo', 'fecha', 'secreto']

Primeras filas:


,dim_0,dim_1,dim_2,dim_3,valor,periodo,fecha,secreto
0,Tasa de paro de la población,Ambos sexos,Total Nacional,Total,11.76,None,2023-10-01T00:00:00.000+02:00,False
1,Tasa de paro de la población,Ambos sexos,Total Nacional,Total,11.84,None,2023-07-01T00:00:00.000+02:00,False
2,Tasa de paro de la población,Ambos sexos,Total Nacional,Total,11.60,None,2023-04-01T00:00:00.000+02:00,False
3,Tasa de paro de la población,Ambos sexos,Total Nacional,Total,13.26,None,2023-01-01T00:00:00.000+01:00,False
4,Tasa de paro de la población,Ambos sexos,Total Nacional,Menores de 25 años,28.36,None,2023-10-01T00:00:00.000+02:00,False
5,Tasa de paro de la población,Ambos sexos,Total Nacional,Menores de 25 años,27.82,None,2023-07-01T00:00:00.000+02:00,False
6,Tasa de paro de la población,Ambos sexos,Total Nacional,Menores de 25 años,27.94,None,2023-04-01T00:00:00.000+02:00,False
7,Tasa de paro de la población,Ambos sexos,Total Nacional,Menores de 25 años,30.03,None,2023-01-01T00:00:00.000+01:00,False
8,Total Nacional,Tasa de paro de la población,Ambos sexos,25 y más años,10.50,None,2023-10-01T00:00:00.000+02:00,False
9,Total Nacional,Tasa de paro de la población,Ambos sexos,25 y más años,10.50,None,2023-07-01T00:00:00.000+02:00,False


In [26]:
#=============================================
#Celda 5 — Análisis rápido de paro
#=============================================
# Ver periodos disponibles
print("Periodos disponibles:")
print(df_paro["periodo"].unique())

print("\nValores nulos por columna:")
print(df_paro.isnull().sum())

print(f"\nRango de valores: {df_paro['valor'].min():.2f}% — {df_paro['valor'].max():.2f}%")
print(f"Media nacional: {df_paro['valor'].mean():.2f}%")


Periodos disponibles:
[None]

Valores nulos por columna:
dim_0         0
dim_1         0
dim_2         0
dim_3         0
valor         6
periodo    1680
fecha         0
secreto       0
dtype: int64

Rango de valores: 2.34% — 100.00%
Media nacional: 21.38%


In [27]:
#=============================================
#Celda 6 — Explorar todas las tablas INE
#=============================================
TABLAS = [
    "tasa_paro_ccaa",
    "renta_media_hogar",
    "densidad_poblacion",
    "turismo_viajeros",
    "turismo_pernoctaciones",
    "centros_educativos_ccaa",
    "ipv_precio_vivienda_ccaa",
    "ipv_precio_vivienda_anual",
    "compraventas_vivienda",
    "hipotecas_vivienda",
]

resumen = []
for tabla in TABLAS:
    raw = data.get(tabla, [])
    if isinstance(raw, list) and len(raw) > 0:
        df_tmp = parse_ine_tabla(raw)
        resumen.append({
            "tabla": tabla,
            "series": len(raw),
            "filas_total": len(df_tmp),
            "columnas": list(df_tmp.columns),
            "periodos": df_tmp["periodo"].nunique() if "periodo" in df_tmp.columns else 0,
            "nulos_%": round(df_tmp["valor"].isnull().mean() * 100, 1) if "valor" in df_tmp.columns else None
        })
    else:
        resumen.append({"tabla": tabla, "series": 0, "filas_total": 0, "error": str(raw)[:80]})

df_resumen = pd.DataFrame(resumen)
print("📊 Resumen de todas las tablas INE:")
df_resumen[["tabla", "series", "filas_total", "periodos", "nulos_%"]]


📊 Resumen de todas las tablas INE:


,tabla,series,filas_total,periodos,nulos_%
0,tasa_paro_ccaa,420,1680,0,0.40
1,renta_media_hogar,32,128,0,0.00
2,densidad_poblacion,159,636,0,0.00
3,turismo_viajeros,420,1680,0,0.00
4,turismo_pernoctaciones,441,1764,0,0.00
5,centros_educativos_ccaa,180,180,0,7.80
6,ipv_precio_vivienda_ccaa,240,896,0,0.00
7,ipv_precio_vivienda_anual,120,448,0,0.00
8,compraventas_vivienda,360,1440,0,0.00
9,hipotecas_vivienda,200,800,0,0.00


In [28]:
#=============================================
#Celda 7 — Explorar Catastro
#=============================================
catastro = data.get("catastro", {})
for ciudad, info in catastro.items():
    print(f"\n🏗️  {ciudad.upper()}")
    print(f"  Status: {info.get('status')}")
    print(f"  Preview: {str(info.get('contenido_preview', ''))[:200]}")



🏗️  MADRID
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cuerr>1</cuerr>
  </control>
  <lerr>
    <err>
      <cod>11</cod>
      <de

🏗️  BARCELONA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cuerr>1</cuerr>
  </control>
  <lerr>
    <err>
      <cod>11</cod>
      <de

🏗️  VALENCIA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cuerr>1</cuerr>
  </control>
  <lerr>
    <err>
      <cod>11</cod>
      <de

🏗️  SEVILLA
  Status: 200
  Preview: <?xml version="1.0" encoding="utf-8"?>
<consulta_municipiero xmlns="http://www.catastro.meh.es/">
  <control>
    <cuerr>1</cuerr>
  </control>
  <lerr>
    <err>
      <cod>11</cod>
      <de


In [29]:
#=============================================
# Celda 8 — Visualización: Tasa de paro por CCAA (último periodo)
#=============================================

# La API INE no devuelve NombrePeriodo, usamos fecha directamente
print(f"Valores únicos en 'periodo': {df_paro['periodo'].unique()}")
print(f"Valores únicos en 'fecha' (muestra): {df_paro['fecha'].dropna().unique()[:5]}")

# Usar fecha como referencia temporal
df_paro["fecha_dt"] = pd.to_datetime(df_paro["fecha"], utc=True, errors="coerce")
ultima_fecha = df_paro["fecha_dt"].max()
print(f"\n📅 Última fecha disponible: {ultima_fecha}")

# Filtrar último periodo
df_paro_ultimo = df_paro[df_paro["fecha_dt"] == ultima_fecha].copy()
df_paro_ultimo = df_paro_ultimo.dropna(subset=["valor"]).sort_values("valor", ascending=True)

print(f"Registros en último periodo: {len(df_paro_ultimo)}")

# Ver columnas de dimensión disponibles
col_dims = [c for c in df_paro_ultimo.columns if c not in ["valor", "periodo", "fecha", "fecha_dt", "secreto"]]
print(f"Columnas dimensión: {col_dims}")
df_paro_ultimo[col_dims + ["valor"]].head(10)


Valores únicos en 'periodo': [None]
Valores únicos en 'fecha' (muestra): <ArrowStringArray>
['2023-10-01T00:00:00.000+02:00', '2023-07-01T00:00:00.000+02:00',
 '2023-04-01T00:00:00.000+02:00', '2023-01-01T00:00:00.000+01:00']
Length: 4, dtype: str

📅 Última fecha disponible: 2023-09-30 22:00:00+00:00
Registros en último periodo: 418
Columnas dimensión: ['dim_0', 'dim_1', 'dim_2', 'dim_3']


,dim_0,dim_1,dim_2,dim_3,valor
1004,Tasa de paro de la población,"Navarra, Comunidad Foral de",Hombres,De 55 y más años,3.91
752,Tasa de paro de la población,Cantabria,Hombres,De 55 y más años,3.99
976,Tasa de paro de la población,"Murcia, Región de",Hombres,De 55 y más años,4.36
1028,Tasa de paro de la población,País Vasco,Hombres,De 25 a 54 años,4.60
468,Tasa de paro de la población,País Vasco,Ambos sexos,De 25 a 54 años,5.02
636,Tasa de paro de la población,Aragón,Hombres,De 25 a 54 años,5.04
624,Tasa de paro de la población,Aragón,Hombres,25 y más años,5.08
1016,Tasa de paro de la población,País Vasco,Hombres,25 y más años,5.11
640,Tasa de paro de la población,Aragón,Hombres,De 55 y más años,5.19
456,Tasa de paro de la población,País Vasco,Ambos sexos,25 y más años,5.46


In [30]:
#=============================================
#Celda 9 — Guardar DataFrames parseados para fase limpieza
#=============================================
output_path = Path("../../output/economico")
output_path.mkdir(exist_ok=True)

# Guardar cada tabla como parquet para la fase de limpieza
for tabla in TABLAS:
    raw = data.get(tabla, [])
    if isinstance(raw, list) and len(raw) > 0:
        df_tmp = parse_ine_tabla(raw)
        df_tmp["fuente_tabla"] = tabla
        df_tmp["fecha_captura"] = data.get("timestamp")
        ruta = output_path / f"raw_{tabla}.parquet"
        df_tmp.to_parquet(ruta, index=False)
        print(f"✅ Guardado: {ruta} ({len(df_tmp)} filas)")

print("\n🎉 Todos los parquets guardados en output/")


✅ Guardado: ../../output/economico/raw_tasa_paro_ccaa.parquet (1680 filas)


✅ Guardado: ../../output/economico/raw_renta_media_hogar.parquet (128 filas)
✅ Guardado: ../../output/economico/raw_densidad_poblacion.parquet (636 filas)
✅ Guardado: ../../output/economico/raw_turismo_viajeros.parquet (1680 filas)
✅ Guardado: ../../output/economico/raw_turismo_pernoctaciones.parquet (1764 filas)
✅ Guardado: ../../output/economico/raw_centros_educativos_ccaa.parquet (180 filas)
✅ Guardado: ../../output/economico/raw_ipv_precio_vivienda_ccaa.parquet (896 filas)
✅ Guardado: ../../output/economico/raw_ipv_precio_vivienda_anual.parquet (448 filas)
✅ Guardado: ../../output/economico/raw_compraventas_vivienda.parquet (1440 filas)
✅ Guardado: ../../output/economico/raw_hipotecas_vivienda.parquet (800 filas)

🎉 Todos los parquets guardados en output/


In [31]:
#=============================================
#
#=============================================
